**for large data on hugging face**

In [ ]:
from datasets import load_dataset
import os
import re

# 1. Load the dataset
print("Downloading large movie dataset...")
dataset = load_dataset("AIatMongoDB/embedded_movies", split="train")

# 2. Setup your movie_data folder
output_dir = "./movie_data_large"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 3. Save as individual files (NTCIR Style)
print("Processing and saving files...")
for i, item in enumerate(dataset):
    
    if not item.get('fullplot') or not item.get('title'):
        continue

    title = item['title']
    plot = item['fullplot']

    directors = item.get('directors') or []
    cast = item.get('cast') or []

   
    safe_title = re.sub(r'[^\w\s]', '', title).strip().replace(' ', '_')
    filename = f"{output_dir}/{safe_title}_{i}.txt"

    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"MOVIE TITLE: {title}\n")
            f.write(f"DIRECTORS: {', '.join(directors)}\n")
            f.write(f"CAST: {', '.join(cast)}\n")
            f.write(f"STORY PLOT: {plot}")
    except Exception as e:
        print(f"Skipping {title} due to error: {e}")

print(f"Success! Your library now contains {len(os.listdir(output_dir))} movie files.")

Now using this because Gemini cause a single point of failure when pushing data in pinecone, so we ar currently using-all-mpnet-base-v2 transformer model

Implimenting using Langchain and Llama3 and pinecone vector database

In [ ]:
pip install -U langgraph langchain-pinecone langchain-google-genai pinecone-client

In [ ]:
!pip install -U langchain-community langchain-text-splitters

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install -U langchain-huggingface sentence-transformers

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
import os
import time
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1. SETUP PINECONE KEY
os.environ["PINECONE_API_KEY"] = "your key here"  
P_KEY = os.environ["PINECONE_API_KEY"]
index_name = "movie-index"

# 2. LOAD & SPLIT DATA
print("📂 Loading 1,500 movies from storage...")
if not os.path.exists('./movie_data_large'):
    print("❌ Error: Directory './movie_data_large' not found!")
else:
    loader = DirectoryLoader('./movie_data_large', glob="./*.txt", loader_cls=TextLoader)
    raw_documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    docs = text_splitter.split_documents(raw_documents)
    print(f"✂️ Created {len(docs)} chunks.")

    # 3. INITIALIZE LOCAL EMBEDDINGS
    print("🤖 Initializing Local Transformer (all-mpnet-base-v2)...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # 4. UPLOAD TO PINECONE
    print(f"🚀 Starting Upload to Pinecone Index: {index_name}...")
    BATCH_SIZE = 100 
    try:
        initial_batch = docs[0:BATCH_SIZE]
        print("Syncing Batch 1 (Creating Connection)...")
        vectorstore = PineconeVectorStore.from_documents(
            initial_batch,
            embeddings,
            index_name=index_name,
            pinecone_api_key=P_KEY
        )

        for i in range(BATCH_SIZE, len(docs), BATCH_SIZE):
            batch = docs[i : i + BATCH_SIZE]
            print(f"Syncing Batch {i//BATCH_SIZE + 1} of {len(docs)//BATCH_SIZE + 1}...")
            vectorstore.add_documents(batch)
            time.sleep(1)

        print("✅ SUCCESS: 1,500 movies are now fully indexed in Pinecone!")

    except Exception as e:
        print(f"❌ System Failure: {e}")

* Gimini - flash Not working so moving to Llama3

In [ ]:
import os
import requests
import time
from langchain_pinecone import PineconeVectorStore

# 1. SETUP
G_KEY = input("Enter your NEW Gemini API Key: ")
index_name = "movie-index"
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 2. RETRIEVAL (Top-1 Only)
query = "Suggest a movie where a character struggles with their own modesty or overconfidence."
print(f"🔍 Searching for the absolute best match...")
retrieved_docs = vectorstore.similarity_search(query, k=1)

context_summary = retrieved_docs[0].page_content[:800]

# 3. DIRECT CALL TO FLASH-LITE (Highest Free Quota)
def call_gemini_free_tier(prompt_text, api_key):
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent?key={api_key}"
    payload = {"contents": [{"parts": [{"text": prompt_text}]}]}

    for attempt in range(2):
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            return response.json()['candidates'][0]['content']['parts'][0]['text']
        elif response.status_code == 429:
            print("⏳ Free tier busy. Waiting 65 seconds for quota reset...")
            time.sleep(65)
        else:
            return f"❌ Error {response.status_code}: {response.text}"
    return "❌ Still busy. Try again in a few minutes."

# 4. COMPACT PROMPT
ntcir_prompt = f"""
NTCIR-19 Analysis:
MOVIE PLOT: {context_summary}
USER QUERY: {query}
Does this movie fit? Explain why or why not briefly.
"""

print("🧠 Generating Response...")
result = call_gemini_free_tier(ntcir_prompt, G_KEY)

print("-" * 30)
print("🎬 NTCIR-19 PREDICTION RESULT:")
print("-" * 30)
print(result)

In [ ]:
!pip install groq

In [ ]:
import os
from groq import Groq
from langchain_pinecone import PineconeVectorStore

# 1. AUTHENTICATION
GROQ_API_KEY = "your key here"  

# 2. INITIALIZE RETRIEVER
print("🤖 Connecting to Pinecone Index...")
index_name = "movie-index"
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 3. SEARCH
query = "Suggest a movie where a character struggles with their own modesty or overconfidence."
print(f"🔍 Searching Pinecone Index...")
retrieved_docs = vectorstore.similarity_search(query, k=3)
context_text = "\n\n".join([f"Movie Plot: {doc.page_content}" for doc in retrieved_docs])

# 4. GROQ INFERENCE (Updated to Llama 3.3)
client = Groq(api_key=GROQ_API_KEY)

print("🧠 Generating NTCIR Response via Groq (Llama 3.3-70B)...")
try:
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a professional movie critic. Analyze the plot provided. Focus on themes of modesty vs. overconfidence."
            },
            {
                "role": "user",
                "content": f"CONTEXT:\n{context_text}\n\nUSER REQUEST: {query}"
            }
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.1,
    )

    print("-" * 30)
    print("🎬 NTCIR-19 PREDICTION RESULT:")
    print("-" * 30)
    print(chat_completion.choices[0].message.content)

except Exception as e:
    print(f"❌ Execution Error: {e}")

In [ ]:
import os
from groq import Groq
from langchain_pinecone import PineconeVectorStore

# 1. SETUP
GROQ_API_KEY = "your key here"  
client = Groq(api_key=GROQ_API_KEY)
index_name = "movie-index"
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 2. RETRIEVE
query = "Suggest a movie where a character struggles with their own modesty or overconfidence."
retrieved_docs = vectorstore.similarity_search(query, k=3)
context_text = "\n\n".join([f"SOURCE_{i+1}: {doc.page_content}" for i, doc in enumerate(retrieved_docs)])

# 3. STRUCTURED INFERENCE
print("🧠 Formatting NTCIR-19 Official Response...")
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": """You are an NTCIR-19 Evaluator. Provide answers in the following STATED format:

            ### [MOVIE TITLE]
            - **Theme Focus**: (Modesty or Overconfidence)
            - **Character Analysis**: (2-3 sentences on the struggle)
            - **Source Grounding**: (Quote a specific detail from the provided context)
            - **Confidence Score**: (0.0 to 1.0 based on how well the context matches)

            If the context does not support the answer, return 'STATUS: INSUFFICIENT DATA'."""
        },
        {
            "role": "user",
            "content": f"CONTEXT FROM VECTOR DB:\n{context_text}\n\nUSER QUERY: {query}"
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0, # Deterministic for NTCIR
)

print("-" * 30)
print("🎬 OFFICIAL NTCIR-19 OUTPUT:")
print("-" * 30)
print(chat_completion.choices[0].message.content)

In [ ]:
import os
from groq import Groq
from langchain_pinecone import PineconeVectorStore

# 1. SETUP
GROQ_API_KEY = "your key here"  
client = Groq(api_key=GROQ_API_KEY)
index_name = "movie-index"

# 2. INITIALIZE VECTOR STORE
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 3. THE COMPLEX QUERY
complex_query = "Find a movie where a character's greatest strength becomes their ultimate undoing or tragic flaw."

print(f"🔍 Searching 1,500 movies for complex narrative patterns...")
retrieved_docs = vectorstore.similarity_search(complex_query, k=3)

context_text = ""
for i, doc in enumerate(retrieved_docs):
    context_text += f"SOURCE_{i+1}:\n{doc.page_content}\n---\n"

# 4. STRUCTURED INFERENCE (NTCIR-19 Format)
print("🧠 Analyzing Subtext with Llama 3.3-70B...")
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": """You are an NTCIR-19 Evaluator specializing in Literary Archetypes.
            Identify a movie from the context where a character experiences a 'Tragic Flaw' (Hamartia).

            Format:
            ### [MOVIE TITLE]
            - **The Strength**: (What was the positive trait?)
            - **The Undoing**: (How did that trait cause the downfall?)
            - **Evidence**: (Reference a specific plot point from the context)
            - **Confidence Score**: (0.0 - 1.0)"""
        },
        {
            "role": "user",
            "content": f"CONTEXT:\n{context_text}\n\nQUERY: {complex_query}"
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0,
)

print("-" * 30)
print("🎬 COMPLEX QUERY RESULT:")
print("-" * 30)
print(chat_completion.choices[0].message.content)